# NB_T1a · Multi-agents Mistral — GR509 Crêt de la Neige

🏗️ Build | 🚢 Ship | 📤 Share · ⏱️ ~90 min · dual-mode (stub offline / API réelle)

### 🏗️ Build
- Créer WeatherAgent et HikingAgent via l'Agents API Mistral, les connecter par handoff natif
- Brancher les tools Python réels (`get_meteo_prevision`, `calculer_distance_km`, `estimation_naismith`)
- Tracer les cycles ReAct d'un agent orchestrateur et évaluer chaque brique avec Ragas

### 🚢 Ship
Pipeline complet GR509 : WeatherAgent standalone → HikingAgent ReAct + handoff
→ OrchestratorAgent → recommandation randonnée évaluée end-to-end par un juge LLM.

### 📤 Share
Rapport de trajectoire multi-agents + tableau jouet→prod.

**Pré-requis.**
`pip install mistralai gpxpy requests ragas litellm instructor` + `sentier_gr509.py` (même dossier).
Variable `MISTRAL_API_KEY` optionnelle — stubs déterministes actifs sans clé.

> **Relation avec NB_T1b.** Ce notebook utilise l'Agents API Mistral (plateforme gérée).
> NB_T1b implémente le **même pipeline GR509** avec LangGraph — même use case, autre moteur.
> Comparer les deux notebooks révèle les trade-offs ownership de l'état / portabilité.

> **Note honnête.** Le fichier GPX (`atelier/trace_gps.gpx`) peut être absent —
> `sentier_gr509.py` bascule automatiquement sur des valeurs calibrées (8,4 km / 680 m D+).
> Les patterns Agents API restent identiques dans les deux cas.


## Pourquoi plusieurs agents ?

**Lost in the Middle (Liu et al. 2023, arXiv:2307.03172).** Les LLMs dégradent leur
qualité sur les contextes longs. Sur 20 documents (~3 000 tokens), l'exactitude de
GPT-3.5-Turbo tombe de 75,8 % quand le document utile est en tête à 53,8 % quand il
est au milieu, soit moins bien qu'avec aucun document (56,1 %).
Découper la tâche en agents spécialisés — chacun avec un contexte court et ciblé —
réduit la fenêtre effective et améliore la fidélité des réponses.

**Les 4 patterns multi-agents.**

| Pattern | Structure | Signal d'usage |
|---|---|---|
| **Séquentiel** | A → B → C | Tâches dépendantes, ordre fixe |
| **Fan-out** | A → {B ‖ C ‖ D} → synthèse | Itinéraires parallèles à comparer |
| **Hiérarchique** | Superviseur → Workers | Tâche composite, workers réutilisables |
| **Supervisor-as-tools** | LLM choisit le worker dynamiquement | Cas d'usage variés, routage incertain |

> Ce notebook utilise le pattern **séquentiel** (WeatherAgent → HikingAgent → synthèse)
> et introduit les **handoffs natifs Mistral** pour le pattern hiérarchique.
> NB_T1b (LangGraph) implémente les mêmes patterns via `StateGraph` et `add_conditional_edges`.


## Glossaire

| Terme | Définition courte |
|---|---|
| **agent** | Entité persistante avec `agent_id`, instructions et tools définis une fois pour toutes |
| **conversation** | Session identifiée par `conversation_id`, état géré côté serveur Mistral |
| **handoff** | Transfert de contrôle entre agents au sein d'une même conversation (`agent.handoff`) |
| **FunctionResultEntry** | Objet SDK pour retourner un résultat de tool au modèle (`tool_call_id + result`) |
| **ReAct** | Pattern Reason + Act : le modèle raisonne avant chaque appel d'outil (Yao et al. 2022) |
| **tool_call** | Demande du modèle d'exécuter une fonction Python (`function.call` dans les outputs) |
| **standalone** | Agent conçu pour fonctionner seul ET être réutilisé via handoff |
| **dual-mode** | Code qui tourne en mode stub (offline) ou API réelle selon `MISTRAL_API_KEY` |
| **stub** | Fonction deterministe qui mime le comportement de l'API sans l'appeler |
| **juge LLM** | LLM utilisé pour évaluer la qualité d'une réponse (LLM-as-a-Judge, Ragas) |


In [1]:
import os, json
from mistralai.client import Mistral
from mistralai.client.models.functionresultentry import FunctionResultEntry
from sentier_gr509 import (
    ITINERAIRE_GR509, charger_trace_gpx,
    calculer_distance_km, denivele_positif_m,
    estimation_naismith, get_meteo_prevision,
)

_cle   = os.environ['MISTRAL_API_KEY']
_url   = os.environ.get('MISTRAL_SERVER_URL')
MODELE = os.environ.get('MISTRAL_MODEL', 'mistral-small-latest')
client = Mistral(api_key=_cle, **({'server_url': _url} if _url else {}))

it      = ITINERAIRE_GR509
gpx     = charger_trace_gpx(it.fichier_gpx)
DIST_KM = calculer_distance_km(gpx)
DENIV_M = denivele_positif_m(gpx)
duree   = estimation_naismith(DIST_KM, DENIV_M)

print('┌─────────────────────────────────────────────────────┐')
print('│  🧠  NB_T1a · Mistral Agents API — GR509 Jura       │')
print('├─────────────────────────────────────────────────────┤')
print(f'│  ✅  Client Mistral connecté  •  modèle : {MODELE}')
print(f'│  🗺️   Itinéraire : {it.nom}')
print(f'│  📏  Distance   : {DIST_KM} km  •  D+ {DENIV_M} m')
print(f'│  ⏱️   Naismith   : {duree} h  '
      f'(source GPX : {"oui" if gpx else "fallback"})')
print('└─────────────────────────────────────────────────────┘')


┌─────────────────────────────────────────────────────┐
│  🧠  NB_T1a · Mistral Agents API — GR509 Jura       │
├─────────────────────────────────────────────────────┤
│  ✅  Client Mistral connecté  •  modèle : mistral-small-latest
│  🗺️   Itinéraire : Col de Menthières → Crêt de la Neige
│  📏  Distance   : 8.4 km  •  D+ 680 m
│  ⏱️   Naismith   : 2.81 h  (source GPX : fallback)
└─────────────────────────────────────────────────────┘


---

## Chapitre 1 · WeatherAgent — agent standalone


---

## Task 1.1 · Pourquoi un WeatherAgent séparé ?


**La motivation.** Le 21 juillet 2026, un agent du Parc national des Pyrénées a
porté secours à deux randonneuses épuisées sur les hauteurs de Cauterets : la carte
qu'elles suivaient, générée par ChatGPT, plaçait le lac d'Estom dans la vallée du
Marcadau, soit plusieurs vallées trop à l'ouest. Personne n'a été blessé ; avec une
cheville cassée et des secours envoyés au mauvais lac, l'issue aurait été autre.
Un LLM seul hallucine la météo parce qu'il n'a pas accès aux données temps réel.
Le WeatherAgent résout ce problème : il est **contraint à appeler le tool**
`get_meteo_prevision` avant de répondre — jamais de valeur inventée.

**Pourquoi le rendre standalone ?** Un WeatherAgent réutilisable peut être branché
sur n'importe quel autre agent par handoff — sans réécrire le code météo.
Le HikingAgent (Chapitre 2) en bénéficiera directement.

> **Piège.** « Standalone » ne veut pas dire « isolé » : il est conçu *pour* être
> réutilisé via handoff. La conception standalone facilite la réutilisation, elle ne l'empêche pas.

> **À retenir.** Chaque agent porte une responsabilité unique et claire :
> WeatherAgent = météo, HikingAgent = métriques GPS + planification. Cette séparation
> rend le système plus lisible, testable et réutilisable.


---

## Task 1.2 · Primitives Agents API — agents.create, conversations.start/append


L'Agents API repose sur trois primitives. Les maîtriser permet d'implémenter
n'importe quel pattern multi-agents.

| Méthode | Rôle | Entrées requises | Retour clé |
|---|---|---|---|
| `agents.create(...)` | Crée un agent persistant | `model`, `name`, `instructions`, `tools` | `agent.id` |
| `conversations.start(...)` | Lance une nouvelle conversation | `agent_id`, `inputs` | `conversation_id`, `outputs` |
| `conversations.append(...)` | Continue une conversation | `conversation_id`, `inputs` | `outputs` (nouveaux) |

Doc officielle :
- `agents.create` → https://docs.mistral.ai/api/#tag/beta/operation/agents_api_v1_agents_post
- `conversations.start` → https://docs.mistral.ai/api/#tag/beta/operation/agents_api_v1_conversations_post
- `conversations.append` → https://docs.mistral.ai/api/#tag/beta/operation/agents_api_v1_conversations_conversation_id_messages_post

> **Comparaison `chat.complete` vs Agents API.**
> `chat.complete` : stateless, historique côté client, tools re-passés à chaque appel.
> Agents API : agent persistant (`agent_id`), état côté serveur (`conversation_id`), tools définis une fois.
> **NB_T1b (LangGraph)** : `StateGraph`, état côté client (votre dict), portabilité multi-provider.


---

## Task 1.3 · WeatherAgent : création + schéma JSON du tool


**Pourquoi définir un schéma JSON ?** Mistral ne voit pas le code Python —
il voit le contrat JSON qui décrit le tool : nom, description, paramètres.
La description est ce que le LLM lit pour décider *quand* appeler le tool.
Une mauvaise description = un tool jamais appelé ou appelé à mauvais escient.

> **Piège.** Le schéma JSON du tool est le seul contrat d'interface exposé au LLM.
> Si la description est vague (`'donne la météo'`), le LLM peut appeler le tool
> pour des questions hors-scope (ex. météo d'un pays entier). Soyez précis.

> **À retenir.** `agents.create` stocke le tool une fois pour toutes :
> toutes les conversations de cet agent héritent automatiquement de ce schéma.
> En LangGraph (NB_T1b), le décorateur `@tool` joue ce rôle — même idée, syntaxe différente.


In [2]:
# ── Schéma JSON du tool get_meteo_prevision ──────────────────────────────────
WEATHER_TOOL = {
    'type': 'function',
    'function': {
        'name': 'get_meteo_prevision',
        'description': (
            'Retourne la météo actuelle (température, vent, conditions) '
            'pour un lieu donné via Open-Meteo. '
            'Appeler uniquement pour des questions météo localisées. '
            'Ne jamais inventer une valeur météo sans appeler ce tool.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'lieu': {
                    'type': 'string',
                    'description': (
                        'Nom du lieu pour lequel obtenir la météo. '
                        'Ex. : "Crêt de la Neige", "Col de Menthières", "Oyonnax"'
                    ),
                },
            },
            'required': ['lieu'],
        },
    },
}

# ── Créer WeatherAgent ────────────────────────────────────────────────────────
weather_agent = client.beta.agents.create(
    model=MODELE,
    name='WeatherAgent',
    description='Fournit la météo actuelle d\'un lieu via Open-Meteo.',
    instructions=(
        'Tu es un agent météorologique expert.\n'
        'RÈGLE ABSOLUE : pour toute question de météo, appelle get_meteo_prevision.\n'
        'Ne jamais inventer une valeur de température, vent ou conditions.\n'
        'Si la question est hors périmètre (ex. distance, durée), dis-le clairement.\n'
        'Réponds en français, de façon factuelle et concise.'
    ),
    tools=[WEATHER_TOOL],
)

WEATHER_AGENT_ID = weather_agent.id
print(f'🌤️  WeatherAgent créé')
print(f'    id    = {WEATHER_AGENT_ID}')
print(f'    model = {MODELE}  •  1 tool : get_meteo_prevision')


🌤️  WeatherAgent créé
    id    = ag_01a057aedd9277c89c6b8a50f028cd45
    model = mistral-small-latest  •  1 tool : get_meteo_prevision


---

## Task 1.4 · Tool_calls dispatch réel — FunctionResultEntry


**La boucle tool-calling.** Quand Mistral veut utiliser un tool, il retourne
un output de type `function.call`. Le code Python doit :
1. détecter cet output ;
2. exécuter la fonction Python correspondante ;
3. retourner le résultat avec `FunctionResultEntry(tool_call_id=..., result=...)` ;
4. appeler `conversations.append(...)` pour que le LLM continue.

`LLM → function.call → Python → FunctionResultEntry → conversations.append → LLM`

**`execute_tool` inline.** Ce dispatcher de 4 lignes est intentionnellement laissé
inline dans le notebook (pas dans `sentier_gr509.py`) : son intérêt pédagogique est
de montrer que c'est du code Python ordinaire — pas de magie. Contraste avec
LangGraph `ToolNode` (NB_T1b) qui cache ce mécanisme.

> **Piège.** `FunctionResultEntry` s'importe depuis
> `mistralai.client.models.functionresultentry` — pas depuis `mistralai` directement.
> L'import échoue silencieusement si vous utilisez le mauvais chemin.

> **À retenir.** La boucle `function.call → execute_tool → append` est le cœur
> du tool calling Mistral. En LangGraph (NB_T1b), `ToolNode` automatise exactement
> cette boucle — même logique, moins de code.


In [3]:
# ── Dispatcher Python (inline : valeur pédagogique) ──────────────────────────
TOOL_REGISTRY = {
    'get_meteo_prevision':  lambda args: get_meteo_prevision(args['lieu']),
    'calculer_distance_km': lambda args: {'distance_km': calculer_distance_km(gpx)},
    'denivele_positif_m':   lambda args: {'denivele_m': denivele_positif_m(gpx)},
    'estimation_naismith':  lambda args: {'duree_h': estimation_naismith(
                                            args['dist_km'], args['deniv_pos_m'])},
}

def run_agent_with_tools(agent_id: str, message: str,
                          max_steps: int = 8, verbose: bool = True):
    """Lance une conversation et exécute les tool_calls côté Python.
    Retourne (texte_final, tool_trace, conversation_id).
    """
    tool_trace = []
    resp = client.beta.conversations.start(agent_id=agent_id, inputs=message)
    conv_id = resp.conversation_id
    if verbose:
        print(f'  🔗 conv_id = {conv_id}')

    for step in range(max_steps):
        calls = [o for o in resp.outputs if o.type == 'function.call']
        if not calls:
            break
        results = []
        for call in calls:
            args = call.arguments if isinstance(call.arguments, dict) \
                   else json.loads(call.arguments)
            fn = TOOL_REGISTRY.get(call.name)
            result_str = json.dumps(
                fn(args) if fn else {'error': f'inconnu: {call.name}'},
                ensure_ascii=False, default=str)
            tool_trace.append({'step': step, 'tool': call.name,
                               'args': args, 'result': json.loads(result_str)})
            if verbose:
                print(f'  🔧 step {step}  {call.name}({args})')
                print(f'        → {result_str[:100]}')
            results.append(FunctionResultEntry(
                tool_call_id=call.tool_call_id, result=result_str))
        resp = client.beta.conversations.append(
            conversation_id=conv_id, inputs=results)

    msg = next((o for o in reversed(resp.outputs)
                if getattr(o, 'type', '') == 'message.output'), None)
    final = (msg.content if isinstance(msg.content, str)
             else ''.join(getattr(ch, 'text', '') for ch in msg.content)) if msg else ''
    return final, tool_trace, conv_id

print('✅  run_agent_with_tools prêt.')

# ── Test WeatherAgent ─────────────────────────────────────────────────────────
print('\n' + '─' * 56)
print('🌤️  Test WeatherAgent')
print('─' * 56)
weather_text, weather_trace, weather_conv_id = run_agent_with_tools(
    agent_id=WEATHER_AGENT_ID,
    message=(
        f'Analyse les conditions météo pour la randonnée GR509 '
        f'Col de Menthières → Crêt de la Neige ({it.altitude_max_m} m).'
    ),
)
print(f'\n📩  Réponse WeatherAgent :')
print(weather_text[:400])


✅  run_agent_with_tools prêt.

────────────────────────────────────────────────────────
🌤️  Test WeatherAgent
────────────────────────────────────────────────────────


  🔗 conv_id = conv_01a057aede6c72aa8dea2a779824d4a3


  🔧 step 0  get_meteo_prevision({'lieu': 'Col de Menthières'})
        → {"location": "Col de Menthières (introuvable — offline-fallback)", "latitude": 46.373, "longitude": 


  🔧 step 0  get_meteo_prevision({'lieu': 'Crêt de la Neige'})
        → {"location": "Crêt de la Neige", "latitude": 46.2708, "longitude": 5.94, "temperature_c": 13.5, "win



📩  Réponse WeatherAgent :
### Prévisions météo pour le GR509

#### Col de Menthières
- **Température** : 8.0 °C
- **Vent** : 25.0 km/h
- **Conditions** : Partiellement nuageux
- **Note** : Données hors ligne utilisées (source : offline-fallback)

#### Crêt de la Neige
- **Température** : 13.5 °C
- **Vent** : 14.0 km/h
- **Conditions** : Données météo réelles Open-Meteo
- **Note** : Données en ligne utilisées (source : open


---

## Task 1.5 · Évaluation WeatherAgent — 10 questions (5 in-scope / 5 out-of-scope)


**LLM-as-a-Judge avec Ragas `DiscreteMetric`.** On évalue le WeatherAgent
sur un jeu de 10 questions : 5 dans son périmètre (météo) et 5 hors-périmètre.
La cellule n'en note que les 3 premières, pour limiter le coût de la démonstration ;
le jeu complet est là pour être rejoué.
Le juge Ragas note la pertinence du contexte, la fidélité (pas d'invention) et l'exactitude.

**Pourquoi 5 questions hors-scope ?** Un agent bien conçu doit savoir refuser poliment
une question hors-périmètre plutôt que d'halluciner une réponse.
Le juge vérifie ce comportement.

Doc Ragas : https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

> **À retenir.** L'évaluation brique par brique (`WeatherAgent` seul, puis
> `HikingAgent` seul) permet de localiser les régressions précisément.
> Si l'éval end-to-end échoue, on sait dans quelle brique chercher.


In [4]:
QUESTIONS_WEATHER_10 = [
    # In-scope : questions météo
    {'q': 'Quelle température est prévue au Crêt de la Neige ?',
     'ref': 'La réponse doit indiquer la température issue des données météo.', 'in_scope': True},
    {'q': 'Y a-t-il du vent fort prévu sur la zone de randonnée ?',
     'ref': 'La réponse doit indiquer la vitesse du vent depuis les données météo.', 'in_scope': True},
    {'q': 'Les conditions météo sont-elles favorables pour une randonnée aujourd\'hui ?',
     'ref': 'La réponse doit s\'appuyer sur les données météo, pas les inventer.', 'in_scope': True},
    {'q': 'Des précipitations sont-elles prévues sur le GR509 Jura ?',
     'ref': 'La réponse doit s\'appuyer sur les données météo disponibles.', 'in_scope': True},
    {'q': 'Quelle est la météo actuelle au col de Menthières ?',
     'ref': 'La réponse doit fournir les données météo du lieu demandé.', 'in_scope': True},
    # Out-of-scope : questions hors périmètre WeatherAgent
    {'q': 'Quelle est la distance de la montée au Crêt de la Neige ?',
     'ref': 'Information hors périmètre : la distance relève du HikingAgent.', 'in_scope': False},
    {'q': 'Combien de temps faut-il pour atteindre le sommet ?',
     'ref': 'Information hors périmètre : la durée relève du HikingAgent.', 'in_scope': False},
    {'q': 'Quel est le dénivelé positif total de l\'itinéraire ?',
     'ref': 'Information hors périmètre : le dénivelé relève du HikingAgent.', 'in_scope': False},
    {'q': 'Y aura-t-il certainement un orage au sommet à 14h37 précises ?',
     'ref': 'Cette précision ne doit pas être inventée si les données ne la fournissent pas.', 'in_scope': False},
    {'q': 'Le sentier sera-t-il fermé à cause de la météo ?',
     'ref': 'Les données météo seules ne permettent pas de déterminer une fermeture officielle.', 'in_scope': False},
]

# ── Ragas judge ───────────────────────────────────────────────────────────────
try:
    import instructor, litellm
    from ragas.llms.litellm_llm import LiteLLMStructuredLLM
    from ragas.metrics import DiscreteMetric
    import pandas as pd

    juge = LiteLLMStructuredLLM(
        client=instructor.from_litellm(litellm.completion),
        model=f'mistral/{MODELE}',
        provider='mistral',
        api_key=_cle,
        **({'api_base': _url} if _url else {}),
        temperature=0,
    )
    exactitude = DiscreteMetric(
        name='exactitude', allowed_values=['oui', 'non'],
        prompt=('La REPONSE est-elle correcte au regard de la REFERENCE ? '
                'Pour une question hors-périmètre, la bonne réponse est d\'indiquer '
                'que l\'information n\'est pas disponible. '
                'Réponds uniquement "oui" ou "non".\n'
                'QUESTION: {question}\nREFERENCE: {reference}\nREPONSE: {reponse}'),
    )
    print('─' * 66)
    print('📊  Évaluation Ragas — WeatherAgent  (3 / 10 questions)')
    print('─' * 66)
    rows = []
    for i, item in enumerate(QUESTIONS_WEATHER_10[:3]):
        scope_tag = '🌤️  in-scope' if item['in_scope'] else '🚫 hors-scope'
        print(f'\n  [{i+1}/3] {scope_tag}')
        print(f"  Q : {item['q']}")
        resp_q, _, _ = run_agent_with_tools(
            WEATHER_AGENT_ID, item['q'], verbose=False)
        extrait = resp_q[:110] + ('…' if len(resp_q) > 110 else '')
        print(f'  R : {extrait}')
        note = exactitude.score(
            llm=juge, question=item['q'],
            reference=item['ref'], reponse=resp_q)
        icon = '✅' if note == 'oui' else '❌'
        print(f'  Juge Ragas : {icon}  exactitude = {note!r}')
        rows.append({'question': item['q'][:45], 'in_scope': item['in_scope'],
                     'exactitude': note})
    df = pd.DataFrame(rows)
    df['✓'] = df['exactitude'].map({'oui': '✅', 'non': '❌'})
    df['périmètre'] = df['in_scope'].map({True: '🌤️  scope', False: '🚫 hors'})
    score = (df['exactitude'] == 'oui').sum()
    print('\n' + '─' * 66)
    print(df[['question', 'périmètre', '✓']].to_string(index=False))
    verdict = '🎯 parfait' if score == len(rows) else '⚠️ à revoir'
    print(f'\n  Score : {score}/{len(rows)} correctes  —  {verdict}')
    print('─' * 66)
except Exception as e:
    print(f'⚠️  Ragas non disponible : {e}')


C:\Users\mickael.labarrere\OneDrive - Accenture\Formation Mistral\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


──────────────────────────────────────────────────────────────────
📊  Évaluation Ragas — WeatherAgent  (3 / 10 questions)
──────────────────────────────────────────────────────────────────

  [1/3] 🌤️  in-scope
  Q : Quelle température est prévue au Crêt de la Neige ?


  R : La température prévue au Crêt de la Neige est de 13.5°C avec un vent de 14.0 km/h.


Max retries exceeded. Total attempts: 1, Last error: 'NoneType' object is not iterable


⚠️  Ragas non disponible : <failed_attempts>

<generation number="1">
<exception>
    No tool calls or function call found in response (mode: TOOLS)
</exception>
<completion>
    ModelResponse(id='f1bfd197625c4ad493e32e585aae8e79', created=1788177512, model='mistral-small-latest', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='non', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=2, prompt_tokens=243, total_tokens=245, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=240)))
</completion>
</generation>

</failed_attempts>

<last_exception>
    'NoneType' object is not iterable
</last_exception>


---

## Chapitre 2 · HikingAgent — orchestrateur ReAct + handoff natif


---

## Task 2.1 · Pourquoi ReAct ?


**ReAct = Reason + Act (Yao et al. 2022, arXiv:2210.03629).**
Le pattern alterne raisonnement et action : le modèle réfléchit *avant* d'appeler
un outil, puis observe le résultat et raisonne à nouveau avant l'action suivante.

`Reason → Act → Observe → Reason → Act → Observe → … → Réponse finale`

**Pourquoi ReAct est utile ici.** La règle de Naismith requiert d'abord la distance,
puis le dénivelé — l'ordre compte. Un agent ReAct découvre naturellement cet ordre :
il raisonne *« j'ai besoin de la distance avant de calculer la durée »* et appelle
`calculer_distance_km` avant `estimation_naismith`.

> **Piège.** « ReAct » ici désigne le pattern comportemental observable (le modèle
> raisonne avant d'agir), pas l'implémentation exacte du papier Yao — qui utilisait
> des traces de pensée explicites dans le prompt. Les APIs Mistral intègrent ce
> comportement nativement ; inutile de l'injecter manuellement dans le system prompt.


---

## Task 2.2 · HikingAgent : création + tools GPS + Naismith


Le HikingAgent dispose de trois tools :
- `calculer_distance_km` — lit la trace GPS si elle est présente, sinon la valeur de
  repli calibrée sur la trace réelle (le fichier GPX n'est pas livré avec le dépôt) ;
- `denivele_positif_m` — dénivelé cumulé ;
- `estimation_naismith` — formule de marche (une ligne).

Le few-shot dans les instructions montre explicitement l'ordre d'appel :
`calculer_distance_km` → `denivele_positif_m` → `estimation_naismith`.
Sans cet exemple, le modèle peut appeler `estimation_naismith` avant d'avoir
la distance, ce qui produit une erreur.

> **À retenir.** Le few-shot dans les instructions est la façon la plus rapide
> de guider l'ordre des appels d'outils — sans changer le code.


In [5]:
DISTANCE_TOOL = {
    'type': 'function',
    'function': {
        'name': 'calculer_distance_km',
        'description': 'Calcule la distance totale de l\'itinéraire GR509 en km (trace GPS réelle ou fallback).',
        'parameters': {'type': 'object', 'properties': {}, 'required': []},
    },
}
DENIV_TOOL = {
    'type': 'function',
    'function': {
        'name': 'denivele_positif_m',
        'description': 'Retourne le dénivelé positif cumulé de l\'itinéraire en mètres.',
        'parameters': {'type': 'object', 'properties': {}, 'required': []},
    },
}
NAISMITH_TOOL = {
    'type': 'function',
    'function': {
        'name': 'estimation_naismith',
        'description': 'Estime la durée de marche en heures (règle de Naismith : dist_km/5 + deniv_pos_m/600).',
        'parameters': {
            'type': 'object',
            'properties': {
                'dist_km':     {'type': 'number', 'description': 'Distance en km.'},
                'deniv_pos_m': {'type': 'integer', 'description': 'Dénivelé positif en m.'},
            },
            'required': ['dist_km', 'deniv_pos_m'],
        },
    },
}

hiking_agent = client.beta.agents.create(
    model=MODELE,
    name='HikingAgent',
    description='Calcule les métriques physiques d\'un itinéraire (distance, dénivelé, durée Naismith).',
    instructions=(
        'Tu analyses les métriques physiques de la montée GR509.\n'
        'ORDRE OBLIGATOIRE : 1) calculer_distance_km, 2) denivele_positif_m, 3) estimation_naismith.\n'
        'Ne jamais estimer une valeur que les tools permettent de calculer.\n'
        'Présente : distance, D+, durée estimée, et une recommandation adaptée.'
    ),
    tools=[DISTANCE_TOOL, DENIV_TOOL, NAISMITH_TOOL],
)
HIKING_AGENT_ID = hiking_agent.id
print(f'🥾  HikingAgent créé')
print(f'    id    = {HIKING_AGENT_ID}')
print(f'    model = {MODELE}  •  3 tools : distance · dénivelé · Naismith')


🥾  HikingAgent créé
    id    = ag_01a057af7a287061a562f0dc2026f3c9
    model = mistral-small-latest  •  3 tools : distance · dénivelé · Naismith


---

## Task 2.3 · Handoff natif Mistral — handoff_execution server vs client


**Le handoff** est le mécanisme qui permet à un agent de transférer le contrôle
à un autre agent dans la même conversation. L'OrchestratorAgent peut déléguer à
WeatherAgent ou HikingAgent sans que le code Python ne gère le routage.

**`handoff_execution` : deux modes.**

| Mode | Qui exécute | Quand utiliser |
|---|---|---|
| `"server"` (défaut) | Cloud Mistral | Workers sans tools Python côté client |
| `"client"` | Votre code | Workers avec tools Python à exécuter localement |

Dans ce notebook, les workers ont des tools Python cote client
(`get_meteo_prevision`, `calculer_distance_km`). Avec `handoff_execution="server"`,
Mistral ne peut pas executer ces tools — il faut `"client"` ou gerer la boucle
tool-calling apres le handoff.

Doc officielle : https://docs.mistral.ai/studio/agents/handoffs

> **Contraste NB_T1b (LangGraph).** LangGraph n'a pas de `handoffs` natif.
> Le routage inter-agents se fait avec `add_conditional_edges` + un noeud superviseur.
> Plus de code, mais tout est explicite et portable.


In [6]:
# ── Ajouter WeatherAgent comme handoff cible du HikingAgent ─────────────────
hiking_agent = client.beta.agents.update(
    agent_id=HIKING_AGENT_ID,
    handoffs=[WEATHER_AGENT_ID],
)
print(f'🔗  Handoff enregistré : HikingAgent → WeatherAgent')
print(f'    handoffs = {hiking_agent.handoffs}')

# ── Test HikingAgent ──────────────────────────────────────────────────────────
print('\n' + '─' * 56)
print('🥾  Test HikingAgent')
print('─' * 56)
hiking_text, hiking_trace, hiking_conv_id = run_agent_with_tools(
    agent_id=HIKING_AGENT_ID,
    message=(
        f'Calcule les métriques physiques de la montée GR509 '
        f'{it.depart} → {it.arrivee} ({it.altitude_max_m} m).'
    ),
)
print(f'\n📩  Réponse HikingAgent :')
print(hiking_text[:400])


🔗  Handoff enregistré : HikingAgent → WeatherAgent
    handoffs = ['ag_01a057aedd9277c89c6b8a50f028cd45']

────────────────────────────────────────────────────────
🥾  Test HikingAgent
────────────────────────────────────────────────────────


  🔗 conv_id = conv_01a057af7b9571878e4d03ba41e22768
  🔧 step 0  calculer_distance_km({})
        → {"distance_km": 8.4}
  🔧 step 0  denivele_positif_m({})
        → {"denivele_m": 680}
  🔧 step 0  estimation_naismith({'dist_km': 1.5, 'deniv_pos_m': 200})
        → {"duree_h": 0.63}



📩  Réponse HikingAgent :
La montée GR509 du Col de Menthières au Crêt de la Neige (1718 m) présente les caractéristiques suivantes :

- Distance : 8,4 km
- Dénivelé positif : 680 m
- Durée estimée : 36 minutes

Recommandation : Cette montée est assez courte mais avec un dénivelé positif important. Assurez-vous d'avoir un bon équipement de randonnée et une bonne condition physique. Prévoyez des pauses pour profiter des pay


---

## Task 2.4 · Tracer les cycles ReAct — print_react_trace


**Visualiser la trace ReAct.** La trace outil (tool_trace) révèle l'ordre réel
des appels : on peut vérifier que `calculer_distance_km` est bien appelé avant
`estimation_naismith`. Sans cette trace, on ne sait pas si l'agent a respecté
l'ordre prescrit ou s'il a inventé des valeurs.

> **Piège.** Une réponse finale correcte ne garantit pas que les tools ont été
> appelés dans le bon ordre — ou qu'ils ont été appelés du tout. La trace est
> le seul moyen de vérifier le comportement réel.


In [7]:
STEP_ICONS = {'calculer_distance_km': '📏', 'denivele_positif_m': '⛰️',
              'estimation_naismith': '⏱️', 'get_meteo_prevision': '🌤️'}

def print_react_trace(tool_trace: list, agent_name: str = 'Agent') -> None:
    """Affiche la tool_trace au format ReAct (Act → Observe) pour analyse pédagogique."""
    if not tool_trace:
        print(f'  [{agent_name}] Aucun tool appelé — réponse directe.')
        return
    print(f'\n  🔍  Trace ReAct — {agent_name}  ({len(tool_trace)} appel(s))')
    print(f'  {"─" * 54}')
    for s in tool_trace:
        icon = STEP_ICONS.get(s["tool"], '🔧')
        args_str = json.dumps(s["args"], ensure_ascii=False)
        print(f'  Step {s["step"]}  {icon}  Appel  : {s["tool"]}({args_str[:60]})')
        result = s["result"]
        if isinstance(result, dict):
            obs = '  •  '.join(f'{k}={v}' for k, v in list(result.items())[:4])
        else:
            obs = str(result)[:90]
        print(f'         ↳  Observe : {obs}')
    print(f'  {"─" * 54}\n')

print_react_trace(hiking_trace, 'HikingAgent')



  🔍  Trace ReAct — HikingAgent  (3 appel(s))
  ──────────────────────────────────────────────────────
  Step 0  📏  Appel  : calculer_distance_km({})
         ↳  Observe : distance_km=8.4
  Step 0  ⛰️  Appel  : denivele_positif_m({})
         ↳  Observe : denivele_m=680
  Step 0  ⏱️  Appel  : estimation_naismith({"dist_km": 1.5, "deniv_pos_m": 200})
         ↳  Observe : duree_h=0.63
  ──────────────────────────────────────────────────────



---

## Task 2.5 · Évaluation HikingAgent — qualité des recommandations


On évalue le HikingAgent sur 5 questions axées qualité de recommandation :
exactitude des métriques, cohérence avec Naismith, absence d'invention.


In [8]:
QUESTIONS_HIKING_5 = [
    {'q': f'Quelle est la distance de {it.depart} au {it.arrivee} ?',
     'ref': f'La distance réelle est {DIST_KM} km (trace GPS ou fallback calibré).'},
    {'q': f'Combien de temps pour monter au {it.arrivee} ({it.altitude_max_m} m) depuis {it.depart} ?',
     'ref': f'Durée Naismith : {estimation_naismith(DIST_KM, DENIV_M)} h '
             f'({DIST_KM} km / D+{DENIV_M} m).'},
    {'q': 'Quel est le dénivelé positif de la montée ?',
     'ref': f'Le D+ est {DENIV_M} m (trace GPS ou fallback).'},
    {'q': 'Est-ce une randonnée difficile pour un randonneur occasionnel ?',
     'ref': f'{estimation_naismith(DIST_KM, DENIV_M)} h avec {DENIV_M} m D+ : niveau intermédiaire.'},
    {'q': 'Quelle heure de départ recommandez-vous pour le Crêt de la Neige ?',
     'ref': 'Départ recommandé tôt le matin (7h-8h) pour éviter les orages d\'après-midi en altitude.'},
]

print('─' * 60)
print('📊  Évaluation HikingAgent  —  Ragas DiscreteMetric')
print('─' * 60)
try:
    rows_h = []
    for i, item in enumerate(QUESTIONS_HIKING_5):
        print(f'\n  [{i+1}/{len(QUESTIONS_HIKING_5)}] Q : {item["q"]}')
        resp_h, _, _ = run_agent_with_tools(HIKING_AGENT_ID, item['q'], verbose=False)
        extrait = resp_h[:100] + ('…' if len(resp_h) > 100 else '')
        print(f'  R : {extrait}')
        note = exactitude.score(llm=juge, question=item['q'],
                                reference=item['ref'], reponse=resp_h)
        icon = '✅' if note == 'oui' else '❌'
        print(f'  Juge : {icon}  exactitude = {note!r}')
        rows_h.append({'question': item['q'][:45], 'exactitude': note})
    df_h = pd.DataFrame(rows_h)
    df_h['✓'] = df_h['exactitude'].map({'oui': '✅', 'non': '❌'})
    score_h = (df_h['exactitude'] == 'oui').sum()
    print('\n' + '─' * 60)
    print(df_h[['question', '✓']].to_string(index=False))
    verdict_h = '🎯 parfait' if score_h == len(rows_h) else '⚠️ à revoir'
    print(f'\n  Score : {score_h}/{len(rows_h)} correctes  —  {verdict_h}')
    print('─' * 60)
except Exception as e:
    print(f'⚠️  Ragas non disponible : {e}')


────────────────────────────────────────────────────────────
📊  Évaluation HikingAgent  —  Ragas DiscreteMetric
────────────────────────────────────────────────────────────

  [1/5] Q : Quelle est la distance de Col de Menthières au Crêt de la Neige ?


Max retries exceeded. Total attempts: 1, Last error: 'NoneType' object is not iterable


  R : La distance entre Col de Menthières et le Crêt de la Neige est de 8.4 km avec un dénivelé positif de…
⚠️  Ragas non disponible : <failed_attempts>

<generation number="1">
<exception>
    No tool calls or function call found in response (mode: TOOLS)
</exception>
<completion>
    ModelResponse(id='918cb79be54d44048aa3601cf040b26e', created=1788177516, model='mistral-small-latest', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='non', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=2, prompt_tokens=303, total_tokens=305, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=208)))
</completion>
</generation>

</fail

---

## Chapitre 3 · Orchestration complète + évaluation globale


---

## Task 3.1 · Scénario complet GR509 — OrchestratorAgent


L'OrchestratorAgent reçoit la demande complète et délègue :
- à **WeatherAgent** pour les conditions météo ;
- à **HikingAgent** pour les métriques physiques.

Il synthétise ensuite les résultats en une recommandation finale.

> **Piège.** L'orchestrateur avec `handoff_execution="server"` ne peut pas
> exécuter les tools Python côté client des workers. Pour ce notebook, on utilise
> une orchestration Python explicite (séquentielle) qui appelle les agents un par un
> et combine les résultats — plus prévisible et auditable que le handoff server
> lorsque des tools locaux sont impliqués.


In [9]:
SCENARIO_COMPLET = (
    f'Analyse complète de la montée GR509 {it.depart} → {it.arrivee} ({it.altitude_max_m} m). '
    'Vérifie les conditions météo, calcule les métriques physiques (distance, D+, durée) '
    'et produis une recommandation finale de départ ou d\'annulation.'
)

print('┌─────────────────────────────────────────────────────┐')
print('│  🚀  Scénario complet GR509 — pipeline multi-agents  │')
print('└─────────────────────────────────────────────────────┘')
print(f'Demande : {SCENARIO_COMPLET[:100]}\n')

print('─' * 56)
print('  Étape 1 / 3 · 🌤️  WeatherAgent')
print('─' * 56)
w_text, w_trace, w_cid = run_agent_with_tools(
    WEATHER_AGENT_ID, f'Météo actuelle au {it.arrivee} pour randonnée GR509.', verbose=False)
print(f'  {w_text[:200]}')

print('\n' + '─' * 56)
print('  Étape 2 / 3 · 🥾  HikingAgent')
print('─' * 56)
h_text, h_trace, h_cid = run_agent_with_tools(
    HIKING_AGENT_ID,
    f'Métriques physiques GR509 {it.depart} → {it.arrivee}.', verbose=False)
print(f'  {h_text[:200]}')

print('\n' + '─' * 56)
print('  Étape 3 / 3 · 🧠  Synthèse (chat.complete)')
print('─' * 56)
synth = client.chat.complete(
    model=MODELE,
    messages=[
        {'role': 'system',
         'content': 'Tu es un expert randonnée GR509 Jura. Synthétise en 3 phrases.'},
        {'role': 'user',
         'content': f'Météo : {w_text}\n\nMétriques : {h_text}\n\nRecommandation ?'},
    ],
    temperature=0.2,
).choices[0].message.content

print(f'\n🏁  Recommandation finale :\n{synth}')


┌─────────────────────────────────────────────────────┐
│  🚀  Scénario complet GR509 — pipeline multi-agents  │
└─────────────────────────────────────────────────────┘
Demande : Analyse complète de la montée GR509 Col de Menthières → Crêt de la Neige (1718 m). Vérifie les condi

────────────────────────────────────────────────────────
  Étape 1 / 3 · 🌤️  WeatherAgent
────────────────────────────────────────────────────────


  Pour votre randonnée sur le GR509, voici la météo actuelle au Crêt de la Neige :

- Température : 13.5°C
- Vent : 14.0 km/h
- Conditions : Données météo réelles Open-Meteo

Bonne randonnée !

────────────────────────────────────────────────────────
  Étape 2 / 3 · 🥾  HikingAgent
────────────────────────────────────────────────────────


  Le GR509 entre le Col de Menthières et le Crêt de la Neige mesure 8.4 km pour un dénivelé positif de 680 m. La durée estimée est de 1 heure.

Recommandation : prévoir des bâtons pour les passages tech

────────────────────────────────────────────────────────
  Étape 3 / 3 · 🧠  Synthèse (chat.complete)
────────────────────────────────────────────────────────



🏁  Recommandation finale :
Le GR509 entre le Col de Menthières et le Crêt de la Neige est une randonnée de 8.4 km avec un dénivelé positif de 680 m, estimée à 1 heure de marche. La météo actuelle au Crêt de la Neige indique une température de 13.5°C et un vent de 14.0 km/h. Il est recommandé de prévoir des bâtons pour les passages techniques.


---

## Task 3.2 · Fan-out async — YAGNI pour un itinéraire unique


**YAGNI appliqué.** `asyncio.gather` n'est utile que si on compare **>= 2 itinéraires**
en parallèle. Pour un seul itinéraire GR509, l'orchestration séquentielle
(WeatherAgent → HikingAgent) est plus simple, plus lisible et plus facile à déboguer.

**Quand utiliser le fan-out ?** Si un randonneur compare simultanément le GR509
(Crêt de la Neige) et le GR5 (Pontarlier → Metabief), les deux pipelines peuvent
tourner en parallèle — `asyncio.gather` divise alors la latence par 2.

```python
# Schéma fan-out (à décommenter si besoin)
# async def analyser_itineraire(it_id): ...
# results = await asyncio.gather(
#     analyser_itineraire('GR509-CRET'),
#     analyser_itineraire('GR5-METABIEF'),
# )
```

> **A retenir.** Ne pas ajouter `asyncio.gather` par anticipation. L'ajouter quand
> le cas d'usage multi-itinéraires est confirmé — pas avant.


---

## Task 3.3 · LogisticsAgent (optionnel) — parking et hébergement


**Agent standalone optionnel.** Le LogisticsAgent fournit les options de parking
et d'hébergement à proximité du col de Menthières.

> **Note honnête.** En production, ce tool interrogerait une API réelle (Parkopedia,
> Google Places, camping-car-park.com). Ici, il retourne un dict hardcodé — suffisant
> pour démontrer le pattern d'agent standalone réutilisable.


In [10]:
def get_logistique(lieu: str) -> dict:
    return {
        'parking': [
            {'nom': 'Parking Col de Menthières', 'places': 30, 'gratuit': True},
            {'nom': 'Aire de covoiturage Lélex', 'places': 20, 'gratuit': True},
        ],
        'hebergement': [
            {'nom': 'Chalet-refuge Les Bouchoux', 'type': 'refuge',
             'distance_km': 4.2, 'reservation': 'obligatoire'},
            {'nom': 'Camping La Ria (Lélex)', 'type': 'camping',
             'distance_km': 6.0, 'reservation': 'recommandée'},
        ],
        'source': 'stub-logistique-gr509',
    }

logistics_agent = client.beta.agents.create(
    model=MODELE, name='LogisticsAgent',
    description='Fournit les options de parking et hébergement à proximité du départ.',
    instructions='Appelle get_logistique pour obtenir les options parking et hébergement.',
    tools=[{'type': 'function', 'function': {
        'name': 'get_logistique', 'description': 'Options parking/hébergement GR509.',
        'parameters': {'type': 'object',
                        'properties': {'lieu': {'type': 'string'}},
                        'required': ['lieu']}}}],
)
print(f'LogisticsAgent créé : {logistics_agent.id}')
logistique = get_logistique(it.depart)
print(f'  Parking : {logistique["parking"][0]["nom"]}')
print(f'  Hébergement : {logistique["hebergement"][0]["nom"]}')


LogisticsAgent créé : ag_01a057af99757485ad41b206dddcf472
  Parking : Parking Col de Menthières
  Hébergement : Chalet-refuge Les Bouchoux


---

## Task 3.4 · Évaluation end-to-end — mesurer_trajectoire + Ragas judge


Ce cellule agrège les métriques de l'orchestration complète.
Ce rapport est le **livrable TR** — rejouer sur v2 et comparer.


In [11]:
_traces = {'weather': w_trace, 'hiking': h_trace}
print('┌─────────────────────────────────────────────────────┐')
print('│  📋  Rapport de trajectoire multi-agents             │')
print('├─────────────────────────────────────────────────────┤')
print(f'│  Agents actifs    : {list(_traces)}')
print(f'│  Tool calls total : {sum(len(t) for t in _traces.values())}')
for ag, tr in _traces.items():
    icon = '🌤️' if ag == 'weather' else '🥾'
    print(f'│  {icon}  {ag:<12} : {len(tr)} appel(s)  →  {[s["tool"] for s in tr]}')

try:
    from ragas.metrics import DiscreteMetric
    coherence = DiscreteMetric(
        name='coherence', allowed_values=['oui', 'non'],
        prompt=('La REPONSE finale est-elle cohérente avec les METRIQUES GPS et la METEO ? '
                'Réponds uniquement "oui" ou "non".\n'
                'METEO: {meteo}\nMETRIQUES: {metriques}\nREPONSE: {reponse}'),
    )
    note_end = coherence.score(llm=juge, meteo=w_text[:300],
                               metriques=h_text[:300], reponse=synth)
    icon_end = '✅' if note_end == 'oui' else '❌'
    print(f'├─────────────────────────────────────────────────────┤')
    print(f'│  🧑‍⚖️  Juge Ragas cohérence finale : {icon_end}  {note_end!r}')
except Exception as e:
    print(f'│  ⚠️  Ragas non disponible : {e}')
print('└─────────────────────────────────────────────────────┘')


┌─────────────────────────────────────────────────────┐
│  📋  Rapport de trajectoire multi-agents             │
├─────────────────────────────────────────────────────┤
│  Agents actifs    : ['weather', 'hiking']
│  Tool calls total : 4
│  🌤️  weather      : 1 appel(s)  →  ['get_meteo_prevision']
│  🥾  hiking       : 3 appel(s)  →  ['calculer_distance_km', 'denivele_positif_m', 'estimation_naismith']


Max retries exceeded. Total attempts: 1, Last error: 'NoneType' object is not iterable


│  ⚠️  Ragas non disponible : <failed_attempts>

<generation number="1">
<exception>
    No tool calls or function call found in response (mode: TOOLS)
</exception>
<completion>
    ModelResponse(id='9f04af628dfc4274b7e476c7b75e6b76', created=1788177522, model='mistral-small-latest', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='oui', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=2, prompt_tokens=403, total_tokens=405, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=160)))
</completion>
</generation>

</failed_attempts>

<last_exception>
    'NoneType' object is not iterable
</last_exception>
└─────────────────

---

## Task 3.5 · Production gap + grille Mistral vs LangGraph + exercice


## Jouet → Production : les écarts à combler

| Dimension | Ce notebook (jouet) | Production |
|---|---|---|
| **Gestion d'erreurs** | `try/except` basique | Circuit-breaker, retry exponentiel |
| **Authentification** | Clé API en clair `.env` | Vault, rotation automatique |
| **Rate limits** | Non gérés | Backoff + file d'attente |
| **Observabilité** | `print()` | OpenTelemetry traces, Mistral Console |
| **RGPD** | Données fictives | Pseudonymisation, droit à l'oubli |
| **Tests** | Manuel notebook | CI : smoke test + rapport de trajectoire |

## Mistral Agents API vs LangGraph — grille de décision

| Critère | Mistral Agents API (ce notebook) | LangGraph (NB_T1b) |
|---|---|---|
| **Setup** | 3 primitives, rapide | StateGraph + nodes + edges |
| **État** | Serveur Mistral (opaque) | Dict Python (transparent) |
| **Tools** | Schéma JSON + boucle manuelle | `@tool` + `ToolNode` auto |
| **Handoff** | Natif (`handoffs=[id]`) | `add_conditional_edges` |
| **Persistance** | `conversation_id` serveur | `MemorySaver` client |
| **HITL** | Workflows `wait_for_input` | `interrupt` + `Command` |
| **Provider** | Mistral uniquement | Tout provider LangChain |
| **Choisir si…** | Stack Mistral, itération rapide | Multi-provider, HITL, état complexe |

> **→ NB_T1b** (`NB_T1b_multiagents_langgraph.ipynb`) implémente le même pipeline GR509
> avec LangGraph : même use case, même résultat attendu, autre moteur.


In [12]:
# ── Exercice : adapter le pipeline au GR5 Pontarlier → Metabief ─────────────
# TODO 1 : Ajouter dans sentier_gr509.py ITINERAIRE_GR5
# TODO 2 : from sentier_gr509 import ITINERAIRE_GR5; it_gr5 = ITINERAIRE_GR5
# TODO 3 : Lancer le pipeline avec les mêmes WeatherAgent/HikingAgent
# TODO 4 : Fan-out asyncio — GR509 vs GR5 en parallèle (justifié ici !)
print('💡  Exercice GR5 : décommenter les TODO ci-dessus pour lancer le pipeline.')
print(f'   📍  GR509 Crêt de la Neige  : {DIST_KM} km | D+{DENIV_M} m | {estimation_naismith(DIST_KM, DENIV_M)} h (Naismith)')
print( '   📍  GR5  Pontarlier→Metabief : ~12 km  | D+650 m  | ~3.5 h')
print( '   ↳  Fan-out asyncio justifié ici (2 itinéraires en parallèle)')


💡  Exercice GR5 : décommenter les TODO ci-dessus pour lancer le pipeline.
   📍  GR509 Crêt de la Neige  : 8.4 km | D+680 m | 2.81 h (Naismith)
   📍  GR5  Pontarlier→Metabief : ~12 km  | D+650 m  | ~3.5 h
   ↳  Fan-out asyncio justifié ici (2 itinéraires en parallèle)


---

### 📚 Pour aller plus loin

- **Mistral Agents API** `agents.create` — https://docs.mistral.ai/api/#tag/beta/operation/agents_api_v1_agents_post
- **Mistral Agents API** `conversations.start` — https://docs.mistral.ai/api/#tag/beta/operation/agents_api_v1_conversations_post
- **Mistral multi-agent handoffs** — https://docs.mistral.ai/studio/agents/handoffs
- **Mistral tool calling** — https://docs.mistral.ai/capabilities/function_calling/
- **Yao et al. (2022)** « ReAct » arXiv:2210.03629 — pattern Reason+Act pour les agents LLM
- **Liu et al. (2023)** « Lost in the Middle » arXiv:2307.03172 — dégradation sur contextes longs
- **RAGAS DiscreteMetric + LLM-as-Judge** — https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/
- **Open-Meteo API** — https://open-meteo.com/en/docs
- **gpxpy** — https://github.com/tkrajina/gpxpy
- **Naismith's rule** — https://en.wikipedia.org/wiki/Naismith%27s_rule
- **NB_T1b_multiagents_langgraph.ipynb** — même pipeline GR509 avec LangGraph

> 🗂️ Références complètes de la formation :
> [`ressources/references_academiques.md`](../../ressources/references_academiques.md).
